In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip list | grep pyspark

pyspark                                  3.5.1


In [ ]:
import pyspark.sql.functions as F

from pyspark.sql import SparkSession
from pyspark.sql import Window

In [ ]:
spark = SparkSession.builder\
    .master("local-cluster[2, 1, 2048]")\
    .config("spark.executor.memory", "2g")\
    .config("spark.driver.memory", "2g")\
    .getOrCreate()
spark

In [ ]:
ROOT_DATA_PATH = "drive/MyDrive/data"

In [ ]:
# Parar spark session
# spark.stop()

## Exemplo de Structured Streaming usando Complete Mode

### Estrutura básica do Streaming

In [ ]:
lines = spark.readStream\
    .format('socket')\
    .option('host', 'localhost')\
    .option('port', 10000)\
    .load()
print(type(lines))

lineCounts = lines.select(F.lower(lines['value']).alias('value'))\
    .groupBy('value')\
    .count()
print(type(lineCounts))

def foreach_batch_function(df, epoch_id):
    print(epoch_id)
    df.show()

query = lineCounts.writeStream\
    .outputMode('complete')\
    .foreachBatch(foreach_batch_function)\
    .start()
query.awaitTermination(90)

In [ ]:
print(type(query))

<class 'pyspark.sql.streaming.query.StreamingQuery'>


In [ ]:
query.stop()

### Exemplo word count com Streaming

In [ ]:
lines = spark.readStream\
    .format('socket')\
    .option('host', 'localhost')\
    .option('port', 10000)\
    .load()

words = lines.select(
    F.explode(
        F.split(lines['value'], ' ')
    ).alias('word')
)

wordCounts = words.groupBy('word').count()

def foreach_batch_function(df, epoch_id):
    print(epoch_id)
    df.show()

query = wordCounts.writeStream\
    .outputMode('complete')\
    .foreachBatch(foreach_batch_function)\
    .start()
query.awaitTermination(120)

In [ ]:
query.stop()

## Exemplo de Structured Streaming usando a coluna Timestamp

In [ ]:
lines = spark.readStream\
    .format('socket')\
    .option('host', 'localhost')\
    .option('port', 10000)\
    .option('includeTimestamp', 'true')\
    .load()

words = lines.select(
    'timestamp',
    F.explode(
        F.split(lines['value'], ' ')
    ).alias('word')
)

wordCounts = words.groupBy('timestamp', 'word').count()

def foreach_batch_function(df, epoch_id):
    print(epoch_id)
    print(df.toPandas())

query = wordCounts.writeStream\
    .outputMode('complete')\
    .foreachBatch(foreach_batch_function)\
    .start()
query.awaitTermination(120)

In [ ]:
query.stop()

## Exemplo de Structured Streaming usando Window

In [ ]:
lines = spark.readStream\
    .format('socket')\
    .option('host', 'localhost')\
    .option('port', 10000)\
    .option('includeTimestamp', 'true')\
    .load()

words = lines.select(
    'timestamp',
    F.explode(
        F.split(lines['value'], ' ')
    ).alias('word')
)

wordCounts = words\
    .groupBy(
        F.window('timestamp', '10 seconds', '10 seconds'),
        'word'
    ).count()\
    .orderBy(F.asc('window.start'), F.asc('count'))

def foreach_batch_function(df, epoch_id):
    print(epoch_id)
    print(df.toPandas())

query = wordCounts.writeStream\
    .outputMode('complete')\
    .foreachBatch(foreach_batch_function)\
    .start()
query.awaitTermination(120)

0
Empty DataFrame
Columns: [window, word, count]
Index: []
1
                                       window word  count
0  (2025-12-04 23:00:00, 2025-12-04 23:00:10)    a      2
1  (2025-12-04 23:00:10, 2025-12-04 23:00:20)    b      3
2  (2025-12-04 23:00:20, 2025-12-04 23:00:30)    c      2
3  (2025-12-04 23:00:30, 2025-12-04 23:00:40)    d      1
2
                                       window word  count
0  (2025-12-04 23:00:00, 2025-12-04 23:00:10)    a      2
1  (2025-12-04 23:00:10, 2025-12-04 23:00:20)    b      3
2  (2025-12-04 23:00:20, 2025-12-04 23:00:30)    c      2
3  (2025-12-04 23:00:30, 2025-12-04 23:00:40)    d      1
4  (2025-12-04 23:01:00, 2025-12-04 23:01:10)    e      4


False

In [ ]:
query.stop()

## Exemplo de Structured Streaming usando Update Mode

In [ ]:
lines = spark\
    .readStream\
    .format('socket')\
    .option('host', 'localhost')\
    .option('port', 10000)\
    .option('includeTimestamp', 'true')\
    .load()

# Split the lines into words
words = lines.select(
    'timestamp',
    F.explode(
        F.split(lines['value'], ' ')
    ).alias('word')
)

wordCounts = words\
    .groupBy(
        F.window('timestamp', '10 seconds', '5 seconds'),
        'word'
    ).count()

def foreach_batch_function(df, epoch_id):
    print(epoch_id)
    print(df.toPandas())

query = wordCounts\
    .writeStream\
    .outputMode('update')\
    .foreachBatch(foreach_batch_function)\
    .start()
query.awaitTermination(60)

In [ ]:
query.stop()

## Exemplo de Structured Streaming usando Append Mode

In [ ]:
lines = spark\
    .readStream\
    .format('socket')\
    .option('host', 'localhost')\
    .option('port', 10000)\
    .option('includeTimestamp', 'true')\
    .load()

# Split the lines into words
words = lines.select(
    'timestamp',
    F.explode(
        F.split(lines['value'], ' ')
    ).alias('word')
)

wordCounts = words\
    .withWatermark('timestamp', '10 seconds')\
    .groupBy(
        F.window('timestamp', '10 seconds', '10 seconds'),
        'word'
    ).count()

def foreach_batch_function(df, epoch_id):
    print(epoch_id)
    print(df.toPandas())

query = wordCounts\
    .writeStream\
    .outputMode('append')\
    .foreachBatch(foreach_batch_function)\
    .start()
query.awaitTermination(60)

In [ ]:
query.stop()

## Exemplos de Structured Streaming usando arquivos CSV

#### Exemplo com dados de bitcoin

In [ ]:
df = spark.read.csv(
    f"{ROOT_DATA_PATH}/bigfile.csv",
    schema="Height INTEGER, \
      Input STRING, \
      Output STRING, \
      Sum STRING, \
      Time TIMESTAMP"
)
df.printSchema()

root
 |-- Height: integer (nullable = true)
 |-- Input: string (nullable = true)
 |-- Output: string (nullable = true)
 |-- Sum: string (nullable = true)
 |-- Time: timestamp (nullable = true)



In [ ]:
df.show(3)

+------+--------------------+--------------------+------+-------------------+
|Height|               Input|              Output|   Sum|               Time|
+------+--------------------+--------------------+------+-------------------+
|   546|['1DZTzaBHUDM7T3Q...|['1KAD5EnzzLtrSo2...|['25']|2009-01-15 06:08:20|
|   546|['1KAD5EnzzLtrSo2...|['1KAD5EnzzLtrSo2...|['25']|2009-01-15 06:08:20|
|   546|['1KAD5EnzzLtrSo2...|['1DZTzaBHUDM7T3Q...|['25']|2009-01-15 06:08:20|
+------+--------------------+--------------------+------+-------------------+
only showing top 3 rows



In [ ]:
%%time

df.select(F.sum('Height')).show()

+-------------+
|  sum(Height)|
+-------------+
|2589395531434|
+-------------+

CPU times: user 7.92 ms, sys: 1.62 ms, total: 9.54 ms
Wall time: 40.1 s


In [ ]:
%%time

df.count()

CPU times: user 2.35 ms, sys: 99 µs, total: 2.44 ms
Wall time: 10.1 s


13494203

In [ ]:
inputStream = spark.readStream.csv(
    f"{ROOT_DATA_PATH}/bitcoin_streaming",
    schema="Height INTEGER, \
      Input STRING, \
      Output STRING, \
      Sum STRING, \
      Time TIMESTAMP"
)

inputStream = inputStream.select(F.sum('Height'))
# inputStream = inputStream.select(F.count('Height'))

def foreach_batch_function(df, epoch_id):
    print(epoch_id)
    print(df.toPandas())

# O comando trigger(processingTime='30 seconds') é importante para controlar quando novos micro-lotes são processados.
query = inputStream\
    .writeStream\
    .outputMode('complete')\
    .trigger(processingTime='30 seconds')\
    .foreachBatch(foreach_batch_function)\
    .start()
query.awaitTermination(360)

0
   sum(Height)
0    918053920
1
    sum(Height)
0  802669056223
2
     sum(Height)
0  1638673426838
3
     sum(Height)
0  2589395531434


False

In [ ]:
query.stop()

#### Qual é a quantidade total de eventos de cada anúncio nos últimos 10 segundos? Calcular a cada 10 segundos e use update mode

In [ ]:
inputStream = spark.readStream.csv(
    f"{ROOT_DATA_PATH}/ad_action",
    schema="timestamp TIMESTAMP, \
      user_id STRING, \
      action STRING, \
      adId STRING, \
      campaignId STRING"
)

inputStream = inputStream\
    .groupBy(
        F.window('timestamp', '10 seconds', '10 seconds'),
        'adId'
    ).count()\

def foreach_batch_function(df, epoch_id):
    df = df.orderBy('window.start')
    print(epoch_id)
    print(df.toPandas())

query = inputStream\
    .writeStream\
    .outputMode('update')\
    .foreachBatch(foreach_batch_function)\
    .start()
query.awaitTermination(60)

0
                                        window     adId  count
0   (2023-09-01 02:42:00, 2023-09-01 02:42:10)  adId_06   9840
1   (2023-09-01 02:42:00, 2023-09-01 02:42:10)  adId_05   8681
2   (2023-09-01 02:42:00, 2023-09-01 02:42:10)  adId_09   9212
3   (2023-09-01 02:42:00, 2023-09-01 02:42:10)  adId_07   9507
4   (2023-09-01 02:42:00, 2023-09-01 02:42:10)  adId_08   8164
5   (2023-09-01 02:42:00, 2023-09-01 02:42:10)  adId_10   5374
6   (2023-09-01 02:42:00, 2023-09-01 02:42:10)  adId_04   8840
7   (2023-09-01 02:42:00, 2023-09-01 02:42:10)  adId_03   8241
8   (2023-09-01 02:42:00, 2023-09-01 02:42:10)  adId_01   7864
9   (2023-09-01 02:42:00, 2023-09-01 02:42:10)  adId_02   9187
10  (2023-09-01 02:42:10, 2023-09-01 02:42:20)  adId_04   7975
11  (2023-09-01 02:42:10, 2023-09-01 02:42:20)  adId_02   8281
12  (2023-09-01 02:42:10, 2023-09-01 02:42:20)  adId_03   7535
13  (2023-09-01 02:42:10, 2023-09-01 02:42:20)  adId_06   8682
14  (2023-09-01 02:42:10, 2023-09-01 02:42:20)  adId_

False

In [ ]:
query.stop()

#### Quais são os top 3 anúncios com mais eventos considerando todos os intervalos de janela? Calcule com uma janela de 10 segundos e periodicidade de 10 segundos e use complete mode.



In [ ]:
inputStream = spark.readStream.csv(
    f"{ROOT_DATA_PATH}/ad_action",
    schema="timestamp TIMESTAMP, \
      user_id STRING, \
      action STRING, \
      adId STRING, \
      campaignId STRING"
)

inputStream = inputStream\
    .groupBy(
        F.window('timestamp', '10 seconds', '10 seconds'),
        'adId'
    ).count()\
    .orderBy(F.desc('count'))\
    .limit(3)

def foreach_batch_function(df, epoch_id):
    print(epoch_id)
    print(df.toPandas())

query = inputStream\
    .writeStream\
    .outputMode('complete')\
    .foreachBatch(foreach_batch_function)\
    .start()
query.awaitTermination(60)

0
                                       window     adId  count
0  (2023-09-01 02:42:00, 2023-09-01 02:42:10)  adId_06   9840
1  (2023-09-01 02:42:20, 2023-09-01 02:42:30)  adId_06   9731
2  (2023-09-01 02:42:00, 2023-09-01 02:42:10)  adId_07   9507


False

In [ ]:
query.stop()

#### Quais são os top 3 anúncios com mais eventos dos últimos 10 segundos em cada intervalo de janela? Calcule a cada 10 segundos e use o complete mode

In [ ]:
inputStream = spark.readStream.csv(
    f"{ROOT_DATA_PATH}/ad_action",
    schema="timestamp TIMESTAMP, \
      user_id STRING, \
      action STRING, \
      adId STRING, \
      campaignId STRING"
)

inputStream = inputStream\
    .groupBy(
        F.window('timestamp', '10 seconds', '10 seconds'),
        'adId'
    ).count()

def foreach_batch_function(df, epoch_id):
    window_group = Window.partitionBy('window.start')\
        .orderBy(F.desc('count'))
    df = df.withColumn('rank', F.row_number().over(window_group))\
        .where(F.col('rank') <= 3)\
        .drop('rank')\
        .orderBy(F.asc('window.start'))
    print(epoch_id)
    print(df.toPandas())

query = inputStream\
    .writeStream\
    .outputMode('complete')\
    .foreachBatch(foreach_batch_function)\
    .start()
query.awaitTermination(60)

0
                                        window     adId  count
0   (2023-09-01 02:42:00, 2023-09-01 02:42:10)  adId_06   9840
1   (2023-09-01 02:42:00, 2023-09-01 02:42:10)  adId_07   9507
2   (2023-09-01 02:42:00, 2023-09-01 02:42:10)  adId_09   9212
3   (2023-09-01 02:42:10, 2023-09-01 02:42:20)  adId_06   8682
4   (2023-09-01 02:42:10, 2023-09-01 02:42:20)  adId_07   8289
5   (2023-09-01 02:42:10, 2023-09-01 02:42:20)  adId_02   8281
6   (2023-09-01 02:42:20, 2023-09-01 02:42:30)  adId_06   9731
7   (2023-09-01 02:42:20, 2023-09-01 02:42:30)  adId_08   9096
8   (2023-09-01 02:42:20, 2023-09-01 02:42:30)  adId_03   8993
9   (2023-09-01 02:42:30, 2023-09-01 02:42:40)  adId_06    935
10  (2023-09-01 02:42:30, 2023-09-01 02:42:40)  adId_03    928
11  (2023-09-01 02:42:30, 2023-09-01 02:42:40)  adId_04    900


False

In [ ]:
query.stop()